# 1) Path to Raw, Processed and Presentation Folder:

In [0]:
%run ../common/configuration


# 2) Intialize the Write Data Process:

In [0]:
from pyspark.sql.functions import current_timestamp, col, lit, concat, to_timestamp, sum, when, count
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, rank
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, FloatType


def add_ingestion_date(input_df):
    return input_df.withColumn("ingestion_date", current_timestamp())


def re_arrange_partition_column(input_df, partition_column):
    column_list = [c for c in input_df.schema.names if c != partition_column]
    column_list.append(partition_column)
    return input_df.select(column_list)



# 3) Check if Raw folder is empty or not:

In [0]:
def is_raw_folder_empty(raw_path):
    try:
        listing = dbutils.fs.ls(raw_path)
    except Exception:
        print(f"{raw_path} does not exist yet.")
        return True

    if len(listing) == 0:
        print(f"{raw_path} is empty.")
        return True

    print(f"{raw_path} already has content:")
    for item in listing:
        print(f"  - {item.name}")
    return False


raw_is_empty = is_raw_folder_empty(raw_folder_path)

# 4) Set up API Request:

In [0]:
import requests
import json
import time

API_BASE_URL = "https://api.jolpi.ca/ergast/f1"
PAGE_LIMIT = 500
FIRST_SEASON = 2000
LAST_SEASON = 2026

SEASON_LEVEL_ENDPOINTS = {
    "races": "{season}/races",
    "constructors": "{season}/constructors",
    "drivers": "{season}/drivers",
    "results": "{season}/results",
    "sprint": "{season}/sprint",
    "qualifying": "{season}/qualifying",
    "driver_standings": "{season}/driverstandings",
    "constructor_standings": "{season}/constructorstandings",
}

ROUND_LEVEL_ENDPOINTS = {
    "pit_stops": "{season}/{round}/pitstops",
    "lap_times": "{season}/{round}/laps",
}

GLOBAL_ENDPOINTS = {
    "seasons": "seasons",
    "circuits": "circuits",
    "status": "status",
}

# 5) Fetch Data and Merge data from multiple seasons (if have):

In [0]:
def fetch_paginated(path, max_retries=3, base_wait_seconds=2):
    combined = None
    table_key = None
    record_key = None
    offset = 0

    while True:
        url = f"{API_BASE_URL}/{path}?limit={PAGE_LIMIT}&offset={offset}"

        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                break  
            except requests.exceptions.RequestException as e:
                if attempt == max_retries:
                    print(f"  Failed on {path} (offset={offset}) after {max_retries} attempts: {e}")
                    raise
                wait_seconds = base_wait_seconds * attempt
                print(f"  Error on {path} (offset={offset}): {e} "
                      f"-- retrying in {wait_seconds}s (attempt {attempt}/{max_retries})...")
                time.sleep(wait_seconds)

        payload = response.json()
        mrdata = payload["MRData"]
        total = int(mrdata.get("total", 0))

        if combined is None:
            combined = payload
            table_key = next(k for k in mrdata if k.endswith("Table"))
            record_key = next(k for k, v in mrdata[table_key].items() if isinstance(v, list))
        else:
            combined["MRData"][table_key][record_key].extend(mrdata[table_key][record_key])

        offset += PAGE_LIMIT
        if offset >= total or total == 0:
            break
        time.sleep(0.2)

    return combined


def merge_payload_into(accumulator, fresh_payload):
    mrdata = fresh_payload["MRData"]
    table_key = next(k for k in mrdata if k.endswith("Table"))
    record_key = next(k for k, v in mrdata[table_key].items() if isinstance(v, list))

    if accumulator is None:
        return fresh_payload, table_key, record_key

    accumulator["MRData"][table_key][record_key].extend(mrdata[table_key][record_key])
    return accumulator, table_key, record_key


def finalize_and_save(accumulator, table_key, record_key, folder_name):
    if accumulator is None:
        print(f"  {folder_name}: no data found, skipping")
        return
    accumulator["MRData"]["total"] = str(len(accumulator["MRData"][table_key][record_key]))
    save_json_to_raw(accumulator, f"{folder_name}/{folder_name}.json")


def save_json_to_raw(data, relative_path):
    full_path = f"{raw_folder_path}/{relative_path}"
    dbutils.fs.put(full_path, json.dumps(data), overwrite=True)
    print(f"  saved {full_path}")


def get_rounds_for_season(season):
    races_payload = fetch_paginated(f"{season}/races")
    races = races_payload["MRData"]["RaceTable"]["Races"]
    return sorted(int(r["round"]) for r in races)


def save_all_progress(season_accumulators, round_accumulators):
    print("Saving current progress...")
    for folder_name, (acc, table_key, record_key) in season_accumulators.items():
        finalize_and_save(acc, table_key, record_key, folder_name)
    for folder_name, (acc, table_key, record_key) in round_accumulators.items():
        finalize_and_save(acc, table_key, record_key, folder_name)
    print("Progress saved.")


def import_all_history():
    print("Importing global endpoints (seasons, circuits, status)...")
    for folder_name, path in GLOBAL_ENDPOINTS.items():
        data = fetch_paginated(path)
        save_json_to_raw(data, f"{folder_name}/{folder_name}.json")

    season_accumulators = {folder_name: (None, None, None) for folder_name in SEASON_LEVEL_ENDPOINTS}
    round_accumulators = {folder_name: (None, None, None) for folder_name in ROUND_LEVEL_ENDPOINTS}

    try:
        for season in range(FIRST_SEASON, LAST_SEASON + 1):
            print(f"Importing season {season}...")

            for folder_name, path_template in SEASON_LEVEL_ENDPOINTS.items():
                fresh = fetch_paginated(path_template.format(season=season))
                acc, table_key, record_key = season_accumulators[folder_name]
                acc, table_key, record_key = merge_payload_into(acc, fresh)
                season_accumulators[folder_name] = (acc, table_key, record_key)

            rounds = get_rounds_for_season(season)
            if not rounds:
                print(f"  no rounds found for {season}, skipping pitstops/laps")
                continue

            for rnd in rounds:
                for folder_name, path_template in ROUND_LEVEL_ENDPOINTS.items():
                    fresh = fetch_paginated(path_template.format(season=season, round=rnd))
                    acc, table_key, record_key = round_accumulators[folder_name]
                    acc, table_key, record_key = merge_payload_into(acc, fresh)
                    round_accumulators[folder_name] = (acc, table_key, record_key)

    except Exception as e:
        print(f"Import stopped due to error: {e}")
        save_all_progress(season_accumulators, round_accumulators)
        raise  

    save_all_progress(season_accumulators, round_accumulators)
    print("Historical import complete.")

# 6) Update New Records if Datasets exist:

In [0]:
def load_existing_json(relative_path):
    full_path = f"{raw_folder_path}/{relative_path}"
    try:
        raw_text = dbutils.fs.head(full_path, 1024 * 1024 * 50)
    except Exception:
        return None
    return json.loads(raw_text)


def merge_new_records(existing_data, fresh_data, table_key, record_key):
    existing_records = existing_data["MRData"][table_key][record_key]
    fresh_records = fresh_data["MRData"][table_key][record_key]

    existing_signatures = {json.dumps(r, sort_keys=True) for r in existing_records}
    new_records = [r for r in fresh_records if json.dumps(r, sort_keys=True) not in existing_signatures]

    if new_records:
        merged_data = existing_data
        merged_data["MRData"][table_key][record_key] = existing_records + new_records
        merged_data["MRData"]["total"] = str(len(existing_records) + len(new_records))
        return merged_data, len(new_records)

    return existing_data, 0


def check_and_update_endpoint(relative_json_path, api_path):
    existing_data = load_existing_json(relative_json_path)

    if existing_data is None:
        fresh_data = fetch_paginated(api_path)
        save_json_to_raw(fresh_data, relative_json_path)
        print(f"  new file: {relative_json_path}")
        return

    fresh_data = fetch_paginated(api_path)
    mrdata = fresh_data["MRData"]
    table_key = next(k for k in mrdata if k.endswith("Table"))
    record_key = next(k for k, v in mrdata[table_key].items() if isinstance(v, list))

    merged_data, new_count = merge_new_records(existing_data, fresh_data, table_key, record_key)

    if new_count > 0:
        save_json_to_raw(merged_data, relative_json_path)
        print(f"  {relative_json_path}: added {new_count} new record(s)")
    else:
        print(f"  {relative_json_path}: no new records")


def get_latest_available_season():
    seasons_payload = fetch_paginated("seasons")
    seasons = seasons_payload["MRData"]["SeasonTable"]["Seasons"]
    return max(int(s["season"]) for s in seasons)


def update_existing_raw_data():
    print("Checking for new data to append to existing raw files...")

    for folder_name, path in GLOBAL_ENDPOINTS.items():
        check_and_update_endpoint(f"{folder_name}/{folder_name}.json", path)

    current_season = get_latest_available_season()

    for folder_name, path_template in SEASON_LEVEL_ENDPOINTS.items():
        check_and_update_endpoint(
            f"{folder_name}/{folder_name}.json",
            path_template.format(season=current_season)
        )

    rounds = get_rounds_for_season(current_season)
    for rnd in rounds:
        for folder_name, path_template in ROUND_LEVEL_ENDPOINTS.items():
            check_and_update_endpoint(
                f"{folder_name}/{folder_name}.json",
                path_template.format(season=current_season, round=rnd)
            )

    print("Update check complete.")

# 7) Check whether data is available (update only) or not (download whole dataset):

In [0]:
if raw_is_empty:
    import_all_history()
else:
    update_existing_raw_data()